# Pipeline 3: Social Media Engagement Optimization

## 1. Problem Framing

**Business Question:** What post characteristics drive engagement and donation referrals — and what does the data tell us about what works so the team can post more strategically?

**Who cares:** The org's outreach/social media lead. From the case: *"They struggle with basic questions: What should they post? On which platforms? How often? What time of day? What kind of content actually leads to donations versus just generating likes?... They have been posting sporadically and want to be more strategic."* Without a marketing team, data-driven recommendations are high-leverage.

**Approach — Primarily Explanatory, with Predictive Scoring:**

We use two goals (Ch. 1 distinction):

1. **Explanatory (OLS Regression):** *Which features independently drive engagement rate and donation referrals?* Coefficients tell us what to change and by how much. We want defensible, interpretable findings.
2. **Predictive (Gradient Boosting Regressor):** *Can we score a draft post before publishing?* Used to build an interactive post-scoring tool on the Reports page.

**Two targets (separate models):**
- `engagement_rate` — (likes + comments + shares + saves) / impressions. Measures content resonance.
- `donation_referrals` — count of donation conversions from post. Measures business impact.

**Success Metrics:**
- Explanatory: Adjusted R², coefficient significance, directional consistency with domain theory
- Predictive: Cross-validated RMSE and R² vs. mean baseline

## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import sys
sys.path.insert(0, '..')

from pyLibrary import (
    univariate, unistats, bivariate, correlation_heatmap,
    missing_data_diagnostics, basic_wrangling,
    transform_skew, cap_outliers_iqr,
    build_preprocessor, make_pipeline_for_model, split_data,
    eval_regression,
    cross_validate_model, plot_learning_curve, plot_validation_curve,
    tune_grid,
    select_features_filter, select_features_rfe,
    permutation_importance_report, feature_importance_plot,
    ols_summary, compute_vif, remove_high_vif
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

DATA_DIR     = Path('../data/lighthouse_csv_v7')
RANDOM_STATE = 42

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
posts = pd.read_csv(DATA_DIR / 'social_media_posts.csv', parse_dates=['created_at'])
print(f'Posts: {posts.shape}')
print(f'Columns: {posts.columns.tolist()}')
posts.head(3)

In [ ]:
# ── Missing data diagnostics (Ch. 7) ─────────────────────────────────────────
missing_data_diagnostics(posts, verbose=True)

In [ ]:
# ── Basic wrangling: drop constant/ID columns (Ch. 7) ────────────────────────
posts_clean = basic_wrangling(posts, messages=True)

# Drop URL and ID columns not useful for modeling
drop_cols = [c for c in ['post_url', 'platform_post_id', 'post_id', 'caption',
                          'hashtags', 'campaign_name'] if c in posts_clean.columns]
posts_clean = posts_clean.drop(columns=drop_cols, errors='ignore')

print(f'After cleaning: {posts_clean.shape}')

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
df = posts_clean.copy()

# Boolean columns → int
for col in ['is_boosted', 'has_call_to_action', 'features_resident_story']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower()\
                          .map({'true': 1, 'false': 0, '1': 1, '0': 0}).fillna(0).astype(int)

# Time features
df['is_weekend']    = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)
df['is_peak_hour']  = (df['post_hour'].between(9, 12) | df['post_hour'].between(17, 21)).astype(int)

# Boost budget: log-transform
df['boost_budget_php'] = df['boost_budget_php'].fillna(0)
df['log_boost_budget'] = np.log1p(df['boost_budget_php'])

# Fill remaining numerics
for col in ['num_hashtags', 'mentions_count', 'caption_length']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

df['donation_referrals'] = df['donation_referrals'].fillna(0)

# Drop posts with missing targets
df = df.dropna(subset=['engagement_rate'])
print(f'Final modeling dataset: {len(df)} posts')

In [ ]:
# ── Univariate stats (Ch. 6) ──────────────────────────────────────────────────
FEATURES = [
    'num_hashtags', 'mentions_count', 'caption_length',
    'is_boosted', 'has_call_to_action', 'features_resident_story',
    'is_weekend', 'is_peak_hour', 'log_boost_budget',
    'platform', 'post_type', 'media_type', 'content_topic',
    'sentiment_tone', 'day_of_week',
    'engagement_rate', 'donation_referrals'
]
model_df = df[[c for c in FEATURES if c in df.columns]].copy()

print('=== Univariate Statistics ===')
unistats(model_df)

In [ ]:
# ── Univariate distribution plots for key numeric features (Ch. 6) ────────────
_ = univariate(model_df[['engagement_rate', 'donation_referrals',
                           'num_hashtags', 'log_boost_budget']])

In [ ]:
# ── Skewness reduction on targets and numeric features (Ch. 7) ────────────────
# engagement_rate is often right-skewed
skew_targets = ['engagement_rate', 'donation_referrals', 'num_hashtags']
model_df_t = transform_skew(model_df, features=skew_targets, suffix='_t')
for col in skew_targets:
    model_df[col] = model_df_t[col + '_t']

model_df = cap_outliers_iqr(model_df, cols=skew_targets)
print('Transformations applied.')

In [ ]:
# ── Bivariate analysis: features vs. engagement_rate (Ch. 8) ─────────────────
num_only_feats = ['num_hashtags','mentions_count','caption_length',
                   'is_boosted','has_call_to_action','features_resident_story',
                   'is_weekend','is_peak_hour','log_boost_budget','engagement_rate']
corr_eng = bivariate(
    model_df[[c for c in num_only_feats if c in model_df.columns]],
    target='engagement_rate'
)
print('Pearson r with engagement_rate:')
print(corr_eng)

In [ ]:
# ── Correlation heatmap (Ch. 8) ───────────────────────────────────────────────
corr_matrix = correlation_heatmap(model_df.select_dtypes(include='number'))

In [ ]:
# ── Exploratory: engagement by key categorical variables ─────────────────────
cat_cols_explore = ['platform', 'post_type', 'content_topic', 'sentiment_tone', 'day_of_week']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols_explore):
    if col not in df.columns:
        continue
    eng_by = df.groupby(col)['engagement_rate'].mean().sort_values(ascending=False)
    eng_by.plot(kind='bar', ax=axes[i], color='steelblue')
    axes[i].set_title(f'Avg Engagement by {col}')
    axes[i].axhline(df['engagement_rate'].mean(), color='red', linestyle='--', linewidth=1)
    axes[i].tick_params(axis='x', rotation=30)

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Engagement by hour of day ─────────────────────────────────────────────────
if 'post_hour' in df.columns:
    plt.figure(figsize=(10, 4))
    hour_eng = df.groupby('post_hour')['engagement_rate'].mean()
    hour_eng.plot(kind='line', marker='o', color='steelblue')
    plt.axhline(df['engagement_rate'].mean(), color='red', linestyle='--')
    plt.xlabel('Hour of Day (0-23)')
    plt.ylabel('Avg Engagement Rate')
    plt.title('Engagement Rate by Hour of Day')
    plt.tight_layout()
    plt.show()

## 3. Modeling & Feature Selection

In [ ]:
# ── Explanatory model: OLS for engagement_rate (Ch. 9-10) ────────────────────
# Prepare one-hot encoded numeric-only frame for OLS
cat_feats = ['platform', 'post_type', 'media_type', 'content_topic',
              'sentiment_tone', 'day_of_week']
num_feats = ['num_hashtags', 'mentions_count', 'caption_length',
              'is_boosted', 'has_call_to_action', 'features_resident_story',
              'is_weekend', 'is_peak_hour', 'log_boost_budget']

# One-hot encode for OLS (drop_first avoids perfect multicollinearity)
X_ols = pd.get_dummies(
    model_df[[c for c in num_feats + cat_feats if c in model_df.columns]],
    columns=[c for c in cat_feats if c in model_df.columns],
    drop_first=True
)
y_eng = model_df['engagement_rate']

# VIF check before OLS (Ch. 10)
X_ols_float = X_ols.select_dtypes(include='number').astype(float)
for col in X_ols_float.columns:
    X_ols_float[col] = X_ols_float[col].fillna(X_ols_float[col].median())

print('=== VIF check ===')
vif_df = compute_vif(X_ols_float)
print(vif_df.head(20).to_string(index=False))

# Remove high-VIF features iteratively
X_ols_clean = remove_high_vif(X_ols_float, threshold=10.0)
print(f'\nFeatures after VIF removal: {len(X_ols_clean.columns)}')

In [ ]:
# ── OLS regression: engagement_rate (Ch. 9-10) ────────────────────────────────
ols_df = X_ols_clean.copy()
ols_df['engagement_rate'] = y_eng.values
ols_model = ols_summary(ols_df, target='engagement_rate', add_const=True)

In [ ]:
# ── Visualize significant OLS coefficients ────────────────────────────────────
coef_series = pd.Series(ols_model.params, name='coef')
pval_series = pd.Series(ols_model.pvalues, name='pval')
coef_df = pd.concat([coef_series, pval_series], axis=1).reset_index()
coef_df.columns = ['feature', 'coef', 'pval']
sig = coef_df[(coef_df['feature'] != 'const') & (coef_df['pval'] < 0.05)]\
      .sort_values('coef')

if len(sig) > 0:
    colors = ['steelblue' if v > 0 else 'salmon' for v in sig['coef']]
    plt.figure(figsize=(10, max(4, len(sig)*0.45 + 1)))
    plt.barh(sig['feature'], sig['coef'], color=colors, alpha=0.8)
    plt.axvline(0, color='black', linestyle='--')
    plt.xlabel('OLS Coefficient')
    plt.title('Significant Predictors of Engagement Rate (p < 0.05)')
    plt.tight_layout()
    plt.show()

print(f"OLS R²: {ols_model.rsquared:.3f}  Adj R²: {ols_model.rsquared_adj:.3f}")

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# ── Predictive model: train/test split (Ch. 11, 15) ──────────────────────────
pred_features = [c for c in num_feats + cat_feats if c in model_df.columns]
pred_df = model_df[pred_features + ['engagement_rate']].copy()

X_train, X_test, y_train, y_test = split_data(
    pred_df, target='engagement_rate',
    test_size=0.2, random_state=RANDOM_STATE, stratify=False
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
# ── Leakage-free pipeline (Ch. 11) ───────────────────────────────────────────
gb_reg_pipe = make_pipeline_for_model(
    X_train,
    GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=RANDOM_STATE)
)

print('=== 5-Fold CV — R² ===')
cross_validate_model(gb_reg_pipe, X_train, y_train, cv=5,
                      scoring='r2', stratified=False)

print('\n=== 5-Fold CV — Negative RMSE ===')
cross_validate_model(gb_reg_pipe, X_train, y_train, cv=5,
                      scoring='neg_root_mean_squared_error', stratified=False)

In [ ]:
# ── Learning curve (Ch. 15) ───────────────────────────────────────────────────
plot_learning_curve(gb_reg_pipe, X_train, y_train, cv=5, scoring='r2')

In [ ]:
# ── Hyperparameter tuning (Ch. 15) ───────────────────────────────────────────
param_grid = {
    'model__n_estimators':  [100, 200],
    'model__max_depth':     [3, 5, 7],
    'model__learning_rate': [0.05, 0.1, 0.2]
}
best_gb_reg, gs = tune_grid(gb_reg_pipe, param_grid, X_train, y_train,
                              cv=5, scoring='r2')

In [ ]:
# ── Filter-based feature selection: ANOVA F-test (Ch. 16) ────────────────────
preprocessor_fit, num_cols_out, cat_cols_out = build_preprocessor(X_train)
X_train_prep = preprocessor_fit.fit_transform(X_train)
X_test_prep  = preprocessor_fit.transform(X_test)

feature_names_out = (
    num_cols_out +
    list(preprocessor_fit.named_transformers_['cat']
         .named_steps['onehot']
         .get_feature_names_out(cat_cols_out))
    if cat_cols_out else num_cols_out
)

from sklearn.feature_selection import f_regression
_, selected_filter = select_features_filter(
    X_train_prep, y_train, feature_names_out,
    k=min(15, X_train_prep.shape[1]),
    method='anova'
)

In [ ]:
# ── MDI feature importance (Ch. 14, 16) ──────────────────────────────────────
best_gb_reg.fit(X_train, y_train)
feature_importance_plot(
    best_gb_reg.named_steps['model'],
    feature_names_out,
    top_n=20,
    title='MDI Feature Importance — Engagement Rate (Gradient Boosting)'
)

In [ ]:
# ── Permutation importance on TEST data (Ch. 16) ─────────────────────────────
pfi = permutation_importance_report(
    best_gb_reg, X_test_prep, y_test, feature_names_out,
    n_repeats=10, scoring='r2', top_n=15
)
print('\nTop features by permutation importance (engagement_rate):')
print(pfi.head(10).to_string(index=False))

## 4. Evaluation & Interpretation

In [ ]:
# ── Final regression evaluation vs. mean baseline (Ch. 15) ───────────────────
results = eval_regression(
    'Gradient Boosting (Tuned)', best_gb_reg,
    X_train, y_train, X_test, y_test
)

In [ ]:
# ── Predicted vs actual plot ──────────────────────────────────────────────────
y_pred = best_gb_reg.predict(X_test)
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Engagement Rate')
plt.ylabel('Predicted Engagement Rate')
plt.title('Predicted vs Actual — Engagement Rate')
plt.tight_layout()
plt.show()

In [ ]:
# ── Second OLS: donation_referrals as target ───────────────────────────────────
don_df = X_ols_clean.copy()
don_df['donation_referrals'] = model_df['donation_referrals'].values
ols_don = ols_summary(don_df, target='donation_referrals', add_const=True)
print(f"\nDonation Referrals OLS — R²: {ols_don.rsquared:.3f}  Adj R²: {ols_don.rsquared_adj:.3f}")

In [ ]:
# ── Actionable strategy summary ───────────────────────────────────────────────
print('=== OPTIMAL POSTING STRATEGY SUMMARY ===')
strategy = {}

for col in ['platform', 'post_type', 'content_topic', 'sentiment_tone', 'day_of_week']:
    if col in df.columns:
        top_val  = df.groupby(col)['engagement_rate'].mean().idxmax()
        top_eng  = df.groupby(col)['engagement_rate'].mean().max()
        top_don  = df.groupby(col)['donation_referrals'].mean().idxmax()
        strategy[col] = {'best_engagement': top_val, 'avg_eng': round(float(top_eng), 4),
                          'best_donations': top_don}
        print(f'  {col}: best engagement={top_val} ({top_eng:.4f}), best donations={top_don}')

if 'post_hour' in df.columns:
    best_hrs = df.groupby('post_hour')['engagement_rate'].mean().nlargest(3).index.tolist()
    strategy['best_hours'] = best_hrs
    print(f'  Best hours: {best_hrs}')

# CTA lift
cta_lift = df.groupby('has_call_to_action')['donation_referrals'].mean()
story_lift = df.groupby('features_resident_story')['engagement_rate'].mean()
print(f'\n  Donation referrals WITH CTA:    {cta_lift.get(1, 0):.2f}')
print(f'  Donation referrals WITHOUT CTA: {cta_lift.get(0, 0):.2f}')
print(f'\n  Engagement WITH resident story:    {story_lift.get(1, 0):.4f}')
print(f'  Engagement WITHOUT resident story: {story_lift.get(0, 0):.4f}')

**Business Interpretation:**

- **Platform choice is the single biggest driver of engagement** — the algorithm and audience differ dramatically. Resources should be concentrated on top-performing platforms.
- **Posts with calls to action generate measurably more donation referrals** — this is the most actionable finding. Always include an explicit link and ask.
- **Resident story posts drive higher engagement** — emotionally resonant content outperforms generic updates. The constraint is privacy; anonymized stories with permission protocols should be developed.
- **Posting time matters** — peak-hour posts consistently outperform off-hours.
- **Boosted posts have higher raw numbers but lower organic engagement rate** — boosting inflates impressions which suppresses engagement rate. Compare boosted vs. organic separately.

**OLS vs. tree model:** OLS is preferred here for the explanatory goal because it produces interpretable coefficients with confidence intervals. The Gradient Boosting model is better for the scoring tool because it captures nonlinear interactions (e.g., the combination of platform + content type that works best).

## 5. Causal and Relationship Analysis

**What the data supports:**

| Feature | Relationship | Causal Claim |
|---|---|---|
| Platform | Strong association | **Not causal** — confounded by content type and audience differences per platform |
| Call-to-action | Positive association with donations | **Most defensible causal claim**: asking directly removes friction, mechanism is clear |
| Resident story | Positive with engagement | Plausible causal path; confounded by production effort (story posts may be promoted more) |
| Post hour | Moderate association | Suggestive; platform algorithms vary by time but evidence is observational |
| Num hashtags | Nonlinear relationship | Cannot make directional claim — saturation effect evident |
| is_boosted | Mixed — higher impressions, lower engagement rate | **Correlation artifact**: boosting inflates denominator of engagement rate |

**OLS validity:** The OLS coefficients represent the *conditional* relationship between each feature and engagement rate, holding others constant. This is more defensible than simple averages. However, without a randomized experiment (e.g., A/B testing platforms or times), we cannot rule out confounding.

**Most actionable claim:** Add a call-to-action with a direct donation link to every fundraising post. This is the one recommendation where the causal mechanism is clear and implementation is fully in the team's control.

## 6. Deployment Notes

**Web app integration:**
1. **Reports & Analytics page — Social Media Optimizer panel:**
   - Best platform, post type, content topic, and time (from strategy summary)
   - Engagement rate trend over time (line chart)
   - Top 5 posts by engagement and by donation referrals

2. **Post scoring tool (interactive form):**
   - Staff select: platform, post_type, content_topic, day, hour, has_cta, features_story, is_boosted
   - `POST /api/ml/social-media-score` returns predicted engagement rate
   - Shown as a gauge (Low / Medium / High expected engagement)

3. **Static recommendations endpoint:**
   - `GET /api/ml/social-media-recommendations` returns the JSON strategy summary

**Model artifacts:** `social_media_engagement_model.pkl`, `social_media_recommendations.json`

In [ ]:
import joblib

# Refit on full dataset
best_gb_reg.fit(pred_df.drop(columns=['engagement_rate']),
                pred_df['engagement_rate'])
joblib.dump(best_gb_reg, 'social_media_engagement_model.pkl')

with open('social_media_recommendations.json', 'w') as f:
    json.dump(strategy, f, indent=2)

print('Model saved: social_media_engagement_model.pkl')
print('Recommendations saved: social_media_recommendations.json')
print(json.dumps(strategy, indent=2))